In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors


import torch
import numpy as np


X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)


y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)


In [ ]:
# 2. Create TensorDataset objects

from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)



In [ ]:
# 3. Create DataLoaders

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch

# Extract the first batch
train_features, train_labels = next(iter(train_loader))

print(f"Feature batch shape: {train_features.shape}") # Expected: [32, Channels, H, W]
print(f"Labels batch shape: {train_labels.shape}")   # Expected: [32, 1]


In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

# Display first 4 images from the batch
plt.figure(figsize=(10, 4))
for i in range(4):
    plt.subplot(1, 4, i + 1)
    # Permute from (C, H, W) to (H, W, C) for plotting
    img = train_features[i].permute(1, 2, 0).numpy()
    plt.imshow(img.astype(np.uint8) if img.max() > 1 else img)
    plt.title(f"Age: {train_labels[i].item():.0f}")
    plt.axis('off')
plt.show()


In [ ]:
# Task 1: Write your model class here:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AgePredictionModel(nn.Module):
    def __init__(self, input_shape):
        super(AgePredictionModel, self).__init__()
        # input_shape should be: Channels * Height * Width
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_shape, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 64)
        self.fc4 = nn.Linear(64, 1) # Output layer: 1 neuron for regression (age)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x) # No activation for the last layer in regression
        return x


In [ ]:
# Task 2: Write your training loop here:
def train_step(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

def validate_step(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    return total_loss / len(loader)


In [ ]:
# Task 3: Write your validation loop here:
def validate_step(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
# Task 4: Define device, model, loss, optimizer:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


sample_batch, _ = next(iter(train_loader))
input_size = sample_batch.shape[1] * sample_batch.shape[2] * sample_batch.shape[3]


model = AgePredictionModel(input_size).to(device)


criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Task 5: Start training for 20 epochs:
epochs = 20
history = {"train_loss": [], "val_loss": []}

print(f"Starting training on {device}...")

for epoch in range(epochs):
    train_loss = train_step(model, train_loader, criterion, optimizer, device)
    val_loss = validate_step(model, test_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print("Training Complete.")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt


plt.figure(figsize=(10, 6))
plt.plot(range(1, epochs + 1), history["train_loss"], label='Training Loss', marker='o')
plt.plot(range(1, epochs + 1), history["val_loss"], label='Validation Loss', marker='o')

plt.title('Age Prediction Model: Loss Over Epochs (2026)')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error (Loss)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:

model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    preds = model(images)

plt.figure(figsize=(15, 10))
for i in range(8):
    plt.subplot(2, 4, i + 1)


    img = images[i].cpu().permute(1, 2, 0).numpy()



    plt.imshow(img.clip(0, 1)) # Clip to ensure valid RGB range

    actual_age = labels[i].item()
    predicted_age = preds[i].item()


    color = 'green' if abs(actual_age - predicted_age) <= 5 else 'red'

    plt.title(f"Actual: {actual_age:.0f}\nPred: {predicted_age:.1f}", color=color)
    plt.axis('off')

plt.tight_layout()
plt.show()
